> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# Parameter search for the intraday momentum strategy on WIN and WDO

A random search of the eight parameters the owner approved (`docs/brief.md` section 10), run on the **WIN mini index** and the **WDO mini dollar**, scored three ways: WIN alone, WDO alone, and **both with shared parameters** (the worse of the two Sharpe ratios). Development period only (2021-12-01 to 2024-12-30, 770 days); the final test period is **not** loaded.

**Read this before any number**
- **The best of many trials is partly luck.** With about 750 days a Sharpe ratio has a standard error near 0.58, and the best of N unrelated trials is expected to reach roughly `0.58 x sqrt(2 ln N)` with no edge at all: about 2.0 for 500 trials, 2.3 for 3,000 and 2.7 for 50,000. Only the held-out numbers count: the leave-one-year-out check, the transfer from one instrument to the other, and the comparison with the **luck baseline** at the end.
- **Costs are in the loop.** Each side pays the B3 fee **plus one tick of slippage**: WIN R$ 0.25 + R$ 1.00, WDO R$ 1.20 + R$ 5.00 per contract and side. Relative to what a contract controls, WDO's costs are about twice WIN's.
- **The sizing is fixed** at the paper's volatility target of 2% and leverage cap of 4; the margin is R$ 1,000 a contract; each instrument runs on its own R$ 100,000 account.
- **The trials are shared.** The same combinations are evaluated once on each instrument; every mode and every fold is scored from the stored daily returns. A day without a sigma (the first days of a long lookback) counts as a flat day. The per-year scores reuse the full-period equity path, which only affects the rounding to whole contracts.
- **Nothing here is chosen for the final test.** This notebook shows how the search behaves.

In [1]:
import multiprocessing as mp
import os
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from tqdm import tqdm

from b3core.intraday_search import (
    INSTRUMENTS,
    SPACE,
    TrialSet,
    decode,
    encode,
    fold_of,
    leave_one_year_out,
    make_config,
    neighbours,
    portfolio_sharpe,
    run_trial,
    sample_indices,
    scores,
    scramble_bars,
    space_size,
    stack_trials,
)
from b3core.winfut import FINAL_START, load_bars

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 230
pd.options.display.max_columns = 40
pd.options.display.max_colwidth = 110

## 1. Settings

In [2]:
N_TRIALS = int(os.environ.get("N_TRIALS", 3_000))  # the budget (3,000 by default; set the variable to change it)
SEED = 20261116  # the sample of combinations, and (plus one) the scrambled data
SLIPPAGE_TICKS = 1.0  # per side, on top of the B3 fee
MIN_TRADES = 200  # round trips in a training fold (two years); a trial with fewer is not ranked
N_NULL = int(os.environ.get("N_NULL", 500))  # trials of the luck baseline
WORKERS = max(1, min(12, (os.cpu_count() or 2) - 1))
MIN_TRADES_FULL = round(MIN_TRADES * 1.5)  # the same minimum scaled to the three years
MODES = ["WIN", "WDO", "joint"]
print(f"{N_TRIALS:,} trials, seed {SEED}, {WORKERS} worker processes, slippage {SLIPPAGE_TICKS:g} tick a side")

50,000 trials, seed 20261116, 11 worker processes, slippage 1 tick a side


## 2. The parameter space and the costs

In [3]:
def clock(minutes: int) -> str:
    return "session end" if minutes >= 1440 else f"{minutes // 60:02d}:{minutes % 60:02d}"


rows = []
for name, values in SPACE.items():
    shown = [clock(v) if name in ("first_decision", "exit_minute") else v for v in values]
    rows.append({"parameter": name, "values": ", ".join(str(v) for v in shown), "count": len(values)})
space_table = pd.DataFrame(rows).set_index("parameter")
display(space_table)
print(f"{space_size():,} combinations in all; {N_TRIALS:,} sampled ({N_TRIALS / space_size():.2%}).")
print("Fixed at the paper's values: target volatility 2%, leverage cap 4, a 14-day volatility window. Not searched: capital, point values, fees, margin.")

bars = {name: load_bars("../samples", symbol=inst.symbol) for name, inst in INSTRUMENTS.items()}
for frame in bars.values():
    assert frame.index.max() < pd.Timestamp(FINAL_START.date()), "a bar from the final test period was loaded"
days = pd.DatetimeIndex(sorted(set.intersection(*[set(f.index.normalize().unique()) for f in bars.values()])))
opens = {n: f["open"].groupby(f.index.normalize()).first().mean() for n, f in bars.items()}
cost_rows = {}
for name, inst in INSTRUMENTS.items():
    exposure = opens[name] * inst.point_value
    side = inst.fee_per_side + SLIPPAGE_TICKS * inst.tick_reais
    cost_rows[name] = {"point value, R$": inst.point_value, "fee a side, R$": inst.fee_per_side, "one tick, R$": inst.tick_reais,
                       "cost a side, R$": side, "contract exposure, R$": exposure, "round trip, basis points": 2 * side / exposure * 1e4}
display(pd.DataFrame(cost_rows).T)
print(f"{len(days)} common days, {days[0].date()} to {days[-1].date()}; bars: " + ", ".join(f"{n} {len(f):,}" for n, f in bars.items()))

,values,count
parameter,,
lookback,"5, 10, 14, 20, 30, 45, 60, 90",8
band_mult,"0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0",7
trade_freq,"15, 30, 60",3
first_decision,"09:00, 09:30, 10:00, 10:30, 11:00, 12:00",6
exit_minute,"16:00, 16:30, 17:00, 17:30, 18:00, 18:20, session end",7
stop,"vwap_band, current_band, opposite_band, vwap_only",4
stop_check,"grid, every_bar",2
sides,"both, long, short",3


169,344 combinations in all; 50,000 sampled (29.53%).
Fixed at the paper's values: target volatility 2%, leverage cap 4, a 14-day volatility window. Not searched: capital, point values, fees, margin.


,"point value, R$","fee a side, R$","one tick, R$","cost a side, R$","contract exposure, R$","round trip, basis points"
WIN,0.20,0.25,1.00,1.25,"23,623.94",1.06
WDO,10.00,1.20,5.00,6.20,"52,107.08",2.38


770 common days, 2021-12-01 to 2024-12-30; bars: WIN 424,422, WDO 428,220


## 3. The search

Each trial runs one combination on both instruments. The pool uses every core but one. The bar shows the rate and the seconds of the last trial; the table after it gives the timing of the whole run: the wall time, the time per trial, and how much the pool saved.

In [4]:
_STATE: dict = {}


def work(index: int):
    started = time.perf_counter()
    returns, sides = run_trial(_STATE["bars"], decode(index), days, slippage_ticks=_STATE["slippage"])
    return index, returns, sides, time.perf_counter() - started


def search(indices: list[int], data: dict, label: str, slippage: float = SLIPPAGE_TICKS):
    _STATE.update(bars=data, slippage=slippage)
    position = {index: i for i, index in enumerate(indices)}
    results: list = [None] * len(indices)
    seconds = np.zeros(len(indices))
    started = time.perf_counter()
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp.get_context("fork")) as pool:
        futures = [pool.submit(work, index) for index in indices]
        done_seconds: list[float] = []
        with tqdm(total=len(indices), desc=label, unit="trial", mininterval=2.0) as bar:
            for future in as_completed(futures):
                index, returns, sides, took = future.result()
                results[position[index]] = (index, returns, sides)
                seconds[position[index]] = took
                done_seconds.append(took)
                bar.update(1)
                bar.set_postfix(last=f"{took:.2f}s", mean=f"{np.mean(done_seconds):.2f}s", refresh=False)
    wall = time.perf_counter() - started
    timing = pd.Series(
        {
            "trials": len(indices),
            "wall time, s": wall,
            "trials per second": len(indices) / wall,
            "seconds per trial, mean (one core, both instruments)": seconds.mean(),
            "seconds per trial, median": np.median(seconds),
            "seconds per trial, 95th percentile": np.percentile(seconds, 95),
            "seconds per trial, slowest": seconds.max(),
            "time of all trials added up, s": seconds.sum(),
            "speed-up from the pool": seconds.sum() / wall,
            "workers": WORKERS,
        },
        name=label,
    )
    return stack_trials(results, days), seconds, timing


indices = sample_indices(N_TRIALS, SEED)
trials, trial_seconds, timing = search(indices, bars, "search")
display(timing.to_frame())

from pathlib import Path

SAVE = Path("../samples/optimization")  # gitignored: derived from vendor data
SAVE.mkdir(parents=True, exist_ok=True)
np.savez_compressed(SAVE / f"trials_{N_TRIALS}_seed{SEED}.npz", indices=trials.indices, days=days.to_numpy().astype("datetime64[ns]"),
                    seconds=trial_seconds, **{f"ret_{k}": v for k, v in trials.ret.items()}, **{f"sides_{k}": v for k, v in trials.sides.items()})
print(f"trial results saved to {SAVE}/trials_{N_TRIALS}_seed{SEED}.npz")

search:   0%|          | 0/50000 [00:00<?, ?trial/s]


search:   0%|          | 15/50000 [00:02<1:51:30,  7.47trial/s, last=3.46s, mean=2.28s]

search:   0%|          | 30/50000 [00:05<2:26:21,  5.69trial/s, last=2.41s, mean=2.10s]

search:   0%|          | 43/50000 [00:07<2:24:38,  5.76trial/s, last=0.85s, mean=1.92s]

search:   0%|          | 57/50000 [00:09<2:22:55,  5.82trial/s, last=0.94s, mean=1.93s]

search:   0%|          | 73/50000 [00:12<2:15:25,  6.14trial/s, last=1.11s, mean=1.91s]

... (3,635 outputs left out)

search: 100%|█████████▉| 49987/50000 [2:16:49<00:02,  5.72trial/s, last=2.35s, mean=1.81s]

search: 100%|██████████| 50000/50000 [2:16:50<00:00,  6.09trial/s, last=1.64s, mean=1.81s]

,search
trials,"50,000.00"
"wall time, s","8,212.73"
trials per second,6.09
"seconds per trial, mean (one core, both instruments)",1.81
"seconds per trial, median",2.28
"seconds per trial, 95th percentile",2.51
"seconds per trial, slowest",3.66
"time of all trials added up, s","90,307.78"
speed-up from the pool,11.00
workers,11.00


trial results saved to ../samples/optimization/trials_50000_seed20261116.npz


**Seconds per trial by parameter.** Which choices make a trial slow (the every-bar stop and the VWAP-only stop are bar-by-bar state machines; the half-hour grid is vectorised).

In [5]:
params = pd.DataFrame([decode(i) for i in trials.indices])
params["exit_minute"] = params["exit_minute"].map(clock)
params["first_decision"] = params["first_decision"].map(clock)
slow = pd.DataFrame({"stop": params["stop"], "stop_check": params["stop_check"], "seconds": trial_seconds})
display(slow.groupby(["stop", "stop_check"])["seconds"].agg(["count", "mean", "max"]).round(2))

count  mean  max
stop          stop_check                  
current_band  every_bar    6263  2.36 3.34
              grid         6175  0.88 1.69
opposite_band every_bar    6278  2.43 3.34
              grid         6288  0.88 1.63
vwap_band     every_bar    6214  2.36 3.46
              grid         6240  0.88 1.47
vwap_only     every_bar    6289  2.38 3.66
              grid         6253  2.27 3.44

## 4. Reference runs

The paper's rule, and two variants already looked at, scored by the same code and with the same costs (fee and one tick). They are extra runs, not part of the trials.

In [6]:
PAPER = {"lookback": 14, "band_mult": 1.0, "trade_freq": 30, "first_decision": 540, "exit_minute": 1440,
         "stop": "vwap_band", "stop_check": "grid", "sides": "both"}
REFERENCE = {
    "paper (no exit time, half-hour stop)": PAPER,
    "exit at 17:30": PAPER | {"exit_minute": 1050},
    "every-bar stop": PAPER | {"stop_check": "every_bar"},
}
ref_indices = [encode(p) for p in REFERENCE.values()]
refs, _, _ = search(ref_indices, bars, "reference runs")
EVERYTHING = np.ones(len(days), dtype=bool)
FULL = {m: scores(trials, EVERYTHING, m, MIN_TRADES_FULL) for m in MODES}
ref_table = pd.DataFrame({m: scores(refs, EVERYTHING, m, 0) for m in MODES}, index=list(REFERENCE))
ref_table["portfolio 50/50"] = portfolio_sharpe(refs, EVERYTHING)
display(ref_table.rename(columns={"joint": "joint (worse of the two)"}))

reference runs:   0%|          | 0/3 [00:00<?, ?trial/s]


reference runs: 100%|██████████| 3/3 [00:01<00:00,  2.14trial/s, last=1.40s, mean=0.88s]

,WIN,WDO,joint (worse of the two),portfolio 50/50
"paper (no exit time, half-hour stop)",0.46,0.44,0.44,0.58
exit at 17:30,-0.29,0.13,-0.29,-0.10
every-bar stop,0.52,-0.59,-0.59,-0.06


## 5. How the trials scored over the whole development period

Sharpe ratio of the daily net returns, costs included. `joint` is the worse of the two instruments' Sharpe ratios; a trial needs at least 300 round trips on each instrument it is scored on. The vertical line is the paper's rule.

In [7]:
rows = {}
for m in MODES:
    s = FULL[m]
    ok = s[np.isfinite(s)]
    rows[m] = {"trials ranked": len(ok), "share > 0": (ok > 0).mean() * 100, "median": np.median(ok), "95th percentile": np.percentile(ok, 95),
               "99th percentile": np.percentile(ok, 99), "best": ok.max(), "paper's rule": ref_table[m if m != "joint" else "joint"].iloc[0]}
display(pd.DataFrame(rows).T)
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{m}" for m in MODES])
for col, m in enumerate(MODES, start=1):
    s = FULL[m]
    fig.add_trace(go.Histogram(x=s[np.isfinite(s)], nbinsx=50, showlegend=False), row=1, col=col)
    fig.add_vline(x=float(ref_table[m].iloc[0]), line_dash="dash", line_color="black", row=1, col=col)
fig.update_layout(title="Sharpe ratio of the trials (dashed: the paper's rule)", height=340, margin=dict(t=70, b=20))
fig.show()

,trials ranked,share > 0,median,95th percentile,99th percentile,best,paper's rule
WIN,"28,226.00",56.39,0.10,1.04,1.39,2.05,0.46
WDO,"27,927.00",22.27,-0.46,0.50,0.91,1.57,0.44
joint,"27,265.00",14.61,-0.57,0.29,0.56,1.12,0.44


## 6. The best trials of each mode

The ten best by score over the whole period. The scores here are in-sample: they are inflated by the search, which is the point of the next sections.

In [8]:
def describe(index: int) -> str:
    p = decode(index)
    return (f"{p['lookback']}d x{p['band_mult']:g} every {p['trade_freq']}m from {clock(p['first_decision'])} "
            f"exit {clock(p['exit_minute'])} {p['stop']}/{p['stop_check']} {p['sides']}")


portfolio_all = portfolio_sharpe(trials, EVERYTHING)
win_all, wdo_all = scores(trials, EVERYTHING, "WIN", 0), scores(trials, EVERYTHING, "WDO", 0)
round_trips = {n: trials.sides[n].sum(axis=1) / 2 for n in ("WIN", "WDO")}


def top_table(mode: str, count: int = 10) -> pd.DataFrame:
    s = FULL[mode]
    order = np.argsort(np.where(np.isfinite(s), s, -np.inf))[::-1][:count]
    return pd.DataFrame(
        {"score": s[order], "WIN Sharpe": win_all[order], "WDO Sharpe": wdo_all[order], "50/50 Sharpe": portfolio_all[order],
         "WIN trades": round_trips["WIN"][order], "WDO trades": round_trips["WDO"][order],
         "combination": [describe(int(trials.indices[i])) for i in order]},
        index=pd.RangeIndex(1, len(order) + 1, name="rank"),
    )


for m in MODES:
    print(f"Best by {m}")
    display(top_table(m))

Best by WIN


,score,WIN Sharpe,WDO Sharpe,50/50 Sharpe,WIN trades,WDO trades,combination
rank,,,,,,,
1,2.05,2.05,-0.79,0.71,368.00,414.00,20d x1.5 every 60m from 12:00 exit 18:20 current_band/every_bar both
2,1.96,1.96,-0.87,0.60,448.00,487.00,20d x1.5 every 60m from 10:30 exit 18:20 current_band/every_bar both
3,1.93,1.93,0.29,1.40,317.00,348.00,20d x1.5 every 30m from 12:00 exit 18:20 vwap_only/every_bar both
4,1.92,1.92,-0.53,0.86,368.00,395.00,90d x1.5 every 60m from 10:30 exit 18:20 vwap_band/every_bar both
5,1.90,1.90,-0.07,1.13,328.00,354.00,60d x1.5 every 60m from 12:00 exit 18:20 current_band/every_bar both
6,1.89,1.89,-0.98,0.47,448.00,487.00,20d x1.5 every 60m from 10:30 exit session end current_band/every_bar both
7,1.88,1.88,-0.03,1.20,309.00,330.00,30d x1.5 every 30m from 12:00 exit 18:20 vwap_only/every_bar both
8,1.86,1.86,-0.77,0.66,535.00,566.00,20d x1.5 every 60m from 09:00 exit 18:20 current_band/every_bar both
9,1.86,1.86,0.20,1.30,342.00,362.00,20d x1.5 every 60m from 11:00 exit 18:20 vwap_only/every_bar both


Best by WDO


,score,WIN Sharpe,WDO Sharpe,50/50 Sharpe,WIN trades,WDO trades,combination
rank,,,,,,,
1,1.57,-0.11,1.57,1.06,443.00,400.00,14d x1 every 15m from 09:30 exit 18:20 vwap_only/grid long
2,1.56,0.30,1.56,1.37,349.00,345.00,14d x1.25 every 15m from 09:00 exit session end vwap_only/grid long
3,1.53,0.03,1.53,1.14,353.00,338.00,10d x1.25 every 15m from 09:30 exit 18:20 vwap_only/grid long
4,1.51,0.46,1.51,1.40,346.00,324.00,60d x1 every 15m from 10:00 exit 18:20 vwap_only/grid long
5,1.50,-0.13,1.50,1.00,443.00,400.00,14d x1 every 15m from 09:30 exit session end vwap_only/grid long
6,1.47,0.36,1.47,1.32,386.00,371.00,90d x1 every 15m from 09:00 exit 18:20 vwap_only/grid long
7,1.45,0.33,1.45,1.28,385.00,373.00,60d x1 every 15m from 09:30 exit 18:20 vwap_only/grid long
8,1.44,0.26,1.44,1.21,324.00,306.00,30d x1.25 every 15m from 09:30 exit 18:20 vwap_only/grid long
9,1.43,0.08,1.43,1.09,410.00,379.00,45d x1 every 15m from 09:30 exit 18:20 vwap_only/grid long


Best by joint


,score,WIN Sharpe,WDO Sharpe,50/50 Sharpe,WIN trades,WDO trades,combination
rank,,,,,,,
1,1.12,1.18,1.12,1.48,417.00,387.00,20d x1.5 every 15m from 10:30 exit 18:20 vwap_only/grid both
2,1.07,1.07,1.10,1.42,334.00,319.00,10d x1.75 every 15m from 10:30 exit 18:20 vwap_only/grid both
3,1.05,1.05,1.08,1.35,459.00,423.00,90d x1 every 15m from 12:00 exit 18:20 vwap_only/grid both
4,1.01,1.07,1.01,1.36,357.00,351.00,60d x1.5 every 15m from 10:30 exit 18:20 vwap_only/grid both
5,1.01,1.03,1.01,1.46,356.00,359.00,14d x1 every 15m from 12:00 exit 18:20 vwap_band/grid long
6,0.97,0.97,1.09,1.31,480.00,428.00,20d x1.5 every 15m from 10:00 exit 18:20 vwap_only/grid both
7,0.95,0.95,1.00,1.26,485.00,461.00,14d x1.25 every 30m from 10:30 exit 18:20 vwap_only/grid both
8,0.94,0.94,0.94,1.20,530.00,499.00,20d x1.5 every 15m from 09:30 exit 18:20 vwap_only/grid both
9,0.93,1.04,0.93,1.38,356.00,360.00,10d x1 every 15m from 12:00 exit session end vwap_band/grid long


## 7. Leave one year out

The honest test inside the development period. For each calendar year (December 2021 joins 2022): pick the best trial on the **other two years**, with at least 200 round trips there, then score it on the year held out. `median test` is what a random pick would get that year; `percentile` is where the winner's held-out score falls among all trials' held-out scores; `paper's rule` is the paper's configuration on the same year. A winner that is great in training and ordinary or worse held out is the signature of overfitting.

In [9]:
folds = fold_of(days)
loyo = {}
for m in MODES:
    table = leave_one_year_out(trials, m, MIN_TRADES)
    paper_test = [float(scores(refs, folds == y, m, 0)[0]) for y in table["held out"]]
    table["paper's rule"] = paper_test
    table["winner"] = [describe(int(trials.indices[int(w)])) if w == w else "none passes the trade minimum" for w in table["winner"]]
    loyo[m] = table
    print(f"Mode {m}")
    display(table.set_index("held out"))

summary = pd.DataFrame(
    {m: {"mean training score of the winners": t["train"].mean(), "mean held-out score of the winners": t["test"].mean(),
         "mean held-out median trial": t["median test"].mean(), "mean held-out paper's rule": t["paper's rule"].mean(),
         "gap, training minus held out": t["train"].mean() - t["test"].mean()} for m, t in loyo.items()}
).T
print("Summary across the three held-out years")
display(summary)

Mode WIN


,winner,train,test,median test,percentile,paper's rule
held out,,,,,,
2022,20d x1.5 every 60m from 12:00 exit 18:20 current_band/every_bar both,2.52,1.12,-0.03,95.23,0.81
2023,5d x1.5 every 30m from 12:00 exit 18:20 vwap_only/every_bar both,2.27,0.18,0.12,53.38,0.66
2024,14d x1 every 60m from 12:00 exit session end vwap_band/grid both,1.75,0.79,0.37,63.82,-0.07


Mode WDO


,winner,train,test,median test,percentile,paper's rule
held out,,,,,,
2022,10d x1 every 60m from 12:00 exit 18:20 current_band/every_bar long,1.83,-0.13,-0.54,65.80,0.48
2023,10d x1.25 every 15m from 09:30 exit 18:20 vwap_only/grid long,2.32,-0.26,-0.75,71.36,0.29
2024,60d x1.5 every 15m from 10:30 exit 18:20 vwap_only/grid both,1.19,0.76,0.24,73.63,0.54


Mode joint


,winner,train,test,median test,percentile,paper's rule
held out,,,,,,
2022,90d x1 every 15m from 12:00 exit 18:20 vwap_only/grid both,1.46,0.01,-0.84,88.36,0.48
2023,60d x1.25 every 60m from 11:00 exit 18:20 vwap_only/grid both,1.24,-0.38,-0.82,71.44,0.29
2024,14d x1 every 15m from 10:30 exit session end vwap_band/grid long,0.91,-0.23,-0.25,50.90,-0.07


Summary across the three held-out years


,mean training score of the winners,mean held-out score of the winners,mean held-out median trial,mean held-out paper's rule,"gap, training minus held out"
WIN,2.18,0.70,0.15,0.47,1.48
WDO,1.78,0.12,-0.35,0.44,1.65
joint,1.20,-0.20,-0.64,0.24,1.40


## 8. Does a winner on one instrument work on the other?

The best trial of WIN, scored on WDO, and the reverse, with each score placed among the other instrument's own scores. The days are the same, so this is not independent in time, but it is a different market with different costs. The joint winner is shown for comparison.

In [10]:
def transfer(source: str, target: str) -> dict:
    s = FULL[source]
    best = int(np.nanargmax(np.where(np.isfinite(s), s, -np.inf)))
    target_scores = scores(trials, EVERYTHING, target, 0) if target != "joint" else FULL["joint"]
    value = target_scores[best]
    finite = target_scores[np.isfinite(target_scores)]
    return {"best trial of": source, "scored on": target, "its score there": value,
            "percentile among that mode's trials": (finite <= value).mean() * 100, "the target's own best": finite.max(),
            "combination": describe(int(trials.indices[best]))}


display(pd.DataFrame([transfer("WIN", "WDO"), transfer("WDO", "WIN"), transfer("joint", "WIN"), transfer("joint", "WDO")]))

,best trial of,scored on,its score there,percentile among that mode's trials,the target's own best,combination
0,WIN,WDO,-0.79,24.01,1.69,20d x1.5 every 60m from 12:00 exit 18:20 current_band/every_bar both
1,WDO,WIN,-0.11,30.58,2.13,14d x1 every 15m from 09:30 exit 18:20 vwap_only/grid long
2,joint,WIN,1.18,95.66,2.13,20d x1.5 every 15m from 10:30 exit 18:20 vwap_only/grid both
3,joint,WDO,1.12,99.39,1.69,20d x1.5 every 15m from 10:30 exit 18:20 vwap_only/grid both


## 9. Which parameters matter

For each value of each parameter, the mean joint score of the trials that use it, and the share of the best 5% of trials (by joint score) that use it against the share in all trials. A parameter whose values all look alike has no effect the search can see; a value that is over-represented among the best is where the search finds something.

In [11]:
joint = FULL["joint"]
ranked = np.isfinite(joint)
top = ranked & (joint >= np.nanpercentile(joint, 95))
raw = pd.DataFrame([decode(i) for i in trials.indices])
def mean_or_nan(values):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(values)


rows = []
for name in SPACE:
    for value in SPACE[name]:
        has = (raw[name] == value).to_numpy()
        rows.append({"parameter": name, "value": clock(value) if name in ("first_decision", "exit_minute") else value,
                     "trials": int(has.sum()), "mean joint score": mean_or_nan(np.where(has & ranked, joint, np.nan)),
                     "share of all trials %": has.mean() * 100, "share of the best 5% %": (has & top).sum() / max(top.sum(), 1) * 100})
effects = pd.DataFrame(rows).set_index(["parameter", "value"])
display(effects)
fig = make_subplots(rows=2, cols=4, subplot_titles=list(SPACE), vertical_spacing=0.18)
for k, name in enumerate(SPACE):
    sub = effects.loc[name]
    fig.add_trace(go.Bar(x=[str(v) for v in sub.index], y=sub["mean joint score"], showlegend=False), row=k // 4 + 1, col=k % 4 + 1)
fig.update_layout(title="Mean joint score by parameter value", height=520, margin=dict(t=80, b=20))
fig.show()

trials  mean joint score  share of all trials %  share of the best 5% %
parameter      value                                                                                  
lookback        5                6308             -0.69                  12.62                    9.68
                10               6210             -0.60                  12.42                   11.66
                14               6307             -0.52                  12.61                   18.26
                20               6191             -0.58                  12.38                   15.76
                30               6213             -0.62                  12.43                    9.09
                45               6235             -0.62                  12.47                    9.38
                60               6184             -0.50                  12.37                   16.57
                90               6352             -0.57                  12.70                    9.60
band_mult      0.50              7134             -0.65                  14.27                    9.46
               0.75              7097             -0.56                  14.19                   14.30
               1.00              7236             -0.47                  14.47                   30.79
               1.25              7095             -0.46                  14.19                   27.27
               1.50              7120             -0.62                  14.24                   13.12
               1.75              7137             -0.85                  14.27                    4.33
               2.00              7181             -1.13                  14.36                    0.73
trade_freq      15              16598             -0.70                  33.20                   36.44
                30              16587             -0.57                  33.17                   35.19
                60              16815             -0.45                  33.63                   28.37
first_decision  09:00            8323             -0.61                  16.65                   17.38
                09:30            8393             -0.58                  16.79                   16.06
                10:00            8276             -0.49                  16.55                   21.92
                10:30            8313             -0.57                  16.63                   18.04
                11:00            8339             -0.69                  16.68                    8.58
                12:00            8356             -0.63                  16.71                   18.04
exit_minute     16:00            7071             -0.69                  14.14                    0.22
                16:30            7153             -0.83                  14.31                    0.15
                17:00            7245             -0.81                  14.49                    0.07
                17:30            7172             -0.73                  14.34                    0.66
                18:00            7136             -0.54                  14.27                    9.02
                18:20            6976             -0.22                  13.95                   55.65
                session end      7247             -0.36                  14.49                   34.24
stop            vwap_band       12454             -0.78                  24.91                   19.87
                current_band    12438             -0.69                  24.88                   22.29
                opposite_band   12566             -0.24                  25.13                   25.22
                vwap_only       12542             -0.47                  25.08                   32.62
stop_check      grid            24956             -0.34                  49.91                   72.36
                every_bar       25044             -0.79                  50.09                   27.64
sides         

## 10. The neighbourhood of the best joint trial

A good combination surrounded by bad neighbours is a warning; a plateau is a good sign. The best joint trial and every combination one step away on one parameter (a step is one value of that parameter's list), run on both instruments with the same costs.

In [12]:
best_joint = int(np.nanargmax(np.where(np.isfinite(joint), joint, -np.inf)))
best_index = int(trials.indices[best_joint])
around = neighbours(best_index)
near, _, near_timing = search([best_index] + around, bars, "neighbourhood")
near_scores = {m: scores(near, EVERYTHING, m, 0) for m in MODES}
table = pd.DataFrame({"WIN Sharpe": near_scores["WIN"], "WDO Sharpe": near_scores["WDO"], "joint score": near_scores["joint"],
                      "changed": ["(the best trial)"] + [", ".join(f"{n} {a} -> {b}" for n, a, b in
                                                                 [(n, decode(best_index)[n], decode(i)[n]) for n in SPACE if decode(i)[n] != decode(best_index)[n]])
                                                         for i in around]})
display(table)
neighbours_joint = near_scores["joint"][1:]
print(f"best joint score {near_scores['joint'][0]:.2f}; neighbours: median {np.nanmedian(neighbours_joint):.2f}, worst {np.nanmin(neighbours_joint):.2f}, "
      f"{(neighbours_joint >= 0.8 * near_scores['joint'][0]).sum()} of {len(neighbours_joint)} keep at least 80% of it")

neighbourhood:   0%|          | 0/13 [00:00<?, ?trial/s]


neighbourhood:  15%|█▌        | 2/13 [00:02<00:14,  1.32s/trial, last=1.09s, mean=1.09s]

neighbourhood: 100%|██████████| 13/13 [00:03<00:00,  3.35trial/s, last=1.25s, mean=2.45s]

,WIN Sharpe,WDO Sharpe,joint score,changed
0,1.18,1.12,1.12,(the best trial)
1,1.02,1.05,1.02,lookback 20 -> 14
2,0.86,0.90,0.86,lookback 20 -> 30
3,0.74,1.00,0.74,band_mult 1.5 -> 1.25
4,1.69,0.77,0.77,band_mult 1.5 -> 1.75
5,1.37,0.87,0.87,trade_freq 15 -> 30
6,0.97,1.09,0.97,first_decision 630 -> 600
7,1.20,1.00,1.00,first_decision 630 -> 660
8,0.78,0.74,0.74,exit_minute 1100 -> 1080
9,1.12,0.96,0.96,exit_minute 1100 -> 1440


best joint score 1.12; neighbours: median 0.87, worst 0.66, 5 of 12 keep at least 80% of it


## 11. The luck baseline

The identical search on the same data with the order of each day's moves **shuffled** (`scramble_bars`): every day keeps its open, its last close, its volume and the set of its moves, but any dependence between one move and the next, and the time-of-day shape of the volatility, is gone, so no momentum edge can exist. The first `N_NULL` of the same combinations are run on both instruments. If the real search's best score is no higher than what pure noise reaches, the best trial is not telling us anything.

In [13]:
scrambled = {name: scramble_bars(frame, SEED + 1) for name, frame in bars.items()}
null_trials, _, null_timing = search(indices[:N_NULL], scrambled, "luck baseline")
display(null_timing.to_frame())
NULL_FULL = {m: scores(null_trials, EVERYTHING, m, MIN_TRADES_FULL) for m in MODES}
rows = {}
for m in MODES:
    real_all = FULL[m]
    real = real_all[np.isfinite(real_all)]
    real_500 = FULL[m][: N_NULL][np.isfinite(FULL[m][: N_NULL])]
    null = NULL_FULL[m][np.isfinite(NULL_FULL[m])]
    rows[m] = {f"real: best of the same {N_NULL:,}": real_500.max(), f"scrambled: best of {N_NULL:,}": null.max(), "real: 95th percentile": np.percentile(real_500, 95),
               "scrambled: 95th percentile": np.percentile(null, 95), "real: median": np.median(real_500), "scrambled: median": np.median(null),
               f"real: best of all {N_TRIALS:,}": real.max()}
display(pd.DataFrame(rows).T)

luck baseline:   0%|          | 0/5000 [00:00<?, ?trial/s]


luck baseline:   0%|          | 7/5000 [00:02<26:37,  3.12trial/s, last=1.11s, mean=1.06s]

luck baseline:   0%|          | 23/5000 [00:05<17:14,  4.81trial/s, last=0.90s, mean=1.63s]

luck baseline:   1%|          | 40/5000 [00:07<13:43,  6.03trial/s, last=0.96s, mean=1.78s]

luck baseline:   1%|          | 53/5000 [00:09<13:44,  6.00trial/s, last=0.84s, mean=1.83s]

luck baseline:   1%|▏         | 66/5000 [00:11<14:11,  5.79trial/s, last=2.52s, mean=1.85s]

... (358 outputs left out)

luck baseline: 100%|█████████▉| 4988/5000 [13:44<00:02,  5.67trial/s, last=2.24s, mean=1.82s]

luck baseline: 100%|██████████| 5000/5000 [13:45<00:00,  6.05trial/s, last=1.72s, mean=1.82s]

,luck baseline
trials,"5,000.00"
"wall time, s",826.40
trials per second,6.05
"seconds per trial, mean (one core, both instruments)",1.82
"seconds per trial, median",2.28
"seconds per trial, 95th percentile",2.54
"seconds per trial, slowest",3.17
"time of all trials added up, s","9,077.42"
speed-up from the pool,10.98
workers,11.00


,"real: best of the same 5,000","scrambled: best of 5,000",real: 95th percentile,scrambled: 95th percentile,real: median,scrambled: median,"real: best of all 50,000"
WIN,1.88,1.36,1.08,0.68,0.09,0.04,2.05
WDO,1.50,0.90,0.50,0.27,-0.46,-0.72,1.57
joint,0.92,0.43,0.29,0.01,-0.57,-0.74,1.12


In [14]:
print("Leave one year out on the scrambled data: what a search finds when there is nothing to find")
null_loyo = {}
for m in MODES:
    t = leave_one_year_out(null_trials, m, MIN_TRADES)
    null_loyo[m] = t
real_rows = {m: {"real: winners' training": loyo[m]["train"].mean(), "real: winners' held out": loyo[m]["test"].mean(),
                 "scrambled: winners' training": null_loyo[m]["train"].mean(), "scrambled: winners' held out": null_loyo[m]["test"].mean()} for m in MODES}
display(pd.DataFrame(real_rows).T)

Leave one year out on the scrambled data: what a search finds when there is nothing to find


,real: winners' training,real: winners' held out,scrambled: winners' training,scrambled: winners' held out
WIN,2.18,0.70,1.58,-0.62
WDO,1.78,0.12,1.24,-1.21
joint,1.20,-0.20,0.62,-0.46


## 12. How to read the results

- **Section 5 and 6** are in-sample and inflated by the search. Compare the best with the 99th percentile and with the luck baseline of section 11.
- **Section 7** is the real test inside the development period: the gap between the winners' training and held-out scores is what the search overfitted. If the held-out score is no better than the median trial or the paper's rule, the search found nothing that lasts.
- **Section 8** asks for a result that survives a change of market, at the cost of independence in time.
- **Section 9 and 10** say whether the winner sits on a plateau and which parameters the search can actually distinguish.
- **Costs:** WDO's costs relative to what a contract controls are about twice WIN's, so the joint score (the worse of the two) is usually set by WDO.
- The final test period has not been read. Nothing here nominates a configuration for it.